# RAG total cost of ownership

Model a RAG product's monthly bill. Three buckets: vector index storage,
embedding compute (one-time + incremental), and LLM inference (the dominant
bucket once you're past prototype).

Prices baked in are illustrative (current as of 2026-05-18) — replace with
your provider's published rates for a real estimate.

In [ ]:
# illustrative USD prices
EMB_PER_M_TOKENS = 0.02       # hosted embedding (input)
LLM_IN_PER_M     = 3.00       # input tokens (uncached)
LLM_OUT_PER_M    = 15.00      # output tokens
CACHED_IN_PER_M  = 0.30       # prompt-cached input tokens (~10% of uncached)
VECTOR_GB_MONTH  = 0.20       # serverless vector DB storage

def rag_monthly_cost(
    docs=1_000_000,          # corpus size in chunks
    chunk_tokens=600,        # tokens per chunk
    daily_queries=1_000_000, # QPS * 86400
    avg_prompt_tokens=2_500, # query + retrieved context + system
    avg_output_tokens=300,
    cache_hit_pct=0.60,      # prompt cache hit fraction
    refresh_pct_per_month=0.05,  # of corpus re-embedded monthly
    dim=1536,                # embedding dim
):
    # storage
    bytes_per_vec = dim * 4 + 100
    storage_gb = docs * bytes_per_vec / 1e9
    storage_usd = storage_gb * VECTOR_GB_MONTH
    # one-time + incremental embedding
    incr_tokens = docs * chunk_tokens * refresh_pct_per_month
    emb_usd = incr_tokens / 1e6 * EMB_PER_M_TOKENS
    # LLM serving
    monthly_q = daily_queries * 30
    cached_in = monthly_q * avg_prompt_tokens * cache_hit_pct
    uncached_in = monthly_q * avg_prompt_tokens * (1 - cache_hit_pct)
    out = monthly_q * avg_output_tokens
    llm_usd = (
        cached_in   / 1e6 * CACHED_IN_PER_M
      + uncached_in / 1e6 * LLM_IN_PER_M
      + out         / 1e6 * LLM_OUT_PER_M
    )
    return {
        'storage':       round(storage_usd),
        'embeddings':    round(emb_usd),
        'llm':           round(llm_usd),
        'total':         round(storage_usd + emb_usd + llm_usd),
        'per_query_cents': round(100 * (storage_usd + emb_usd + llm_usd) / monthly_q, 3),
    }

for scale, dq in [('startup', 10_000), ('series-B', 200_000), ('big', 10_000_000)]:
    r = rag_monthly_cost(daily_queries=dq)
    print(f'{scale:10s} daily queries={dq:>10,d}  '
          f'storage=${r["storage"]:>7,d}  emb=${r["embeddings"]:>6,d}  '
          f'llm=${r["llm"]:>10,d}  total=${r["total"]:>10,d}  '
          f'{r["per_query_cents"]:.3f} cents/query')

Where the money goes (this matters):

1. **LLM inference is 90%+ of the bill** at any non-trivial scale. The other two buckets are rounding error.
2. **Prompt cache hit ratio is the biggest knob.** Going from 0% to 70% cache hits cuts the LLM bill in half on a typical RAG prompt. See `06-llm-serving-and-rag/`.
3. **Output tokens are 5x more expensive than input tokens.** Constrain output length.
4. **Smaller model for the cheap path.** If 60% of queries are simple, route them to a cheaper model.
5. **Don't over-retrieve.** Halving retrieved context halves the input token bill on uncached prompts.